# 07_AE_Robustness_Analysis_Repeated_Stratified_CV

This notebook evaluates the robustness of the proposed CNN autoencoder anomaly scoring methods with respect to different validation and test set compositions. 
Reconstruction-based anomaly scores are computed once using a fixed pretrained model and subsequently assessed using repeated stratified cross-validation on a combined evaluation pool.

The analysis focuses on:

- Global reconstruction error metrics (`global_mse`, `global_mae`)
- Segmented Vertical Error (SVE) anomaly scores (`sve_5`, `sve_7`)
- Threshold stability across evaluation splits
- Variability of Precision, Recall, F1-score, and PR-AUC

The resulting mean and standard deviation of the evaluation metrics provide an estimate of score robustness and sensitivity to different dataset compositions without retraining the underlying autoencoder.

In [103]:
from pathlib import Path
import json
import torch

import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from sklearn.model_selection import StratifiedKFold, RepeatedStratifiedKFold
import matplotlib.pyplot as plt
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    confusion_matrix,
    precision_recall_curve
)

In [104]:
# -----------------------------
# paths
# -----------------------------
REPO_ROOT = Path("../..").resolve()
MANIFEST_PATH = REPO_ROOT / 'reports' / 'manifests' / 'broach_dataset_split_seed42.csv'

MODEL_PATH = REPO_ROOT / "models" / "ae_bn16_broach_dataset_PyTorch_20260709.pth"
THRESHOLD_DIR = REPO_ROOT / 'reports' / 'thresholds'
SCORE_DIR = REPO_ROOT / 'reports' / 'scores'
TABLE_DIR = REPO_ROOT / 'reports' / 'tables'
SVE_PATH = REPO_ROOT / 'reports' / 'SVE'

DATA_ROOT = REPO_ROOT / "data/03_broach_dataset"

IMAGE_SIZE = (150, 100)

# outputs
OUTPUT_DIR = REPO_ROOT / "reports"
OUTPUT_DIR.mkdir(exist_ok=True)

In [105]:
class Autoencoder(torch.nn.Module):

    def __init__(self):
        super().__init__()

        # Encoder
        self.encoder_conv = torch.nn.Sequential(
            torch.nn.Conv2d(3, 4, kernel_size=3, padding=1),
            torch.nn.ReLU(),

            torch.nn.MaxPool2d(kernel_size=2, stride=2),

            torch.nn.Conv2d(4, 8, kernel_size=3, padding=1),
            torch.nn.ReLU(),

            torch.nn.MaxPool2d(kernel_size=(2, 3), stride=(2, 3)),

            torch.nn.Conv2d(8, 12, kernel_size=3, padding=1),
            torch.nn.ReLU()
        )

        self.flatten = torch.nn.Flatten()

        self.encoder_fc = torch.nn.Sequential(
            torch.nn.Linear(12 * 25 * 25, 16),
            torch.nn.ReLU()
        )

        # Decoder
        self.decoder_fc = torch.nn.Sequential(
            torch.nn.Linear(16, 12 * 25 * 25),
            torch.nn.ReLU()
        )

        self.decoder_conv = torch.nn.Sequential(
            torch.nn.Conv2d(12, 12, kernel_size=3, padding=1),
            torch.nn.ReLU(),

            torch.nn.Upsample(scale_factor=(2, 3), mode='nearest'),

            torch.nn.Conv2d(12, 8, kernel_size=3, padding=1),
            torch.nn.ReLU(),

            torch.nn.Upsample(scale_factor=(2, 2), mode='nearest'),

            torch.nn.Conv2d(8, 4, kernel_size=3, padding=1),
            torch.nn.ReLU(),

            torch.nn.Conv2d(4, 3, kernel_size=3, padding=1),
            torch.nn.Sigmoid()
        )

    def forward(self, x):

        x = self.encoder_conv(x)

        x = self.flatten(x)

        latent = self.encoder_fc(x)

        x = self.decoder_fc(latent)

        x = x.view(-1, 12, 25, 25)

        x = self.decoder_conv(x)

        return x

In [4]:
# Load model
model = Autoencoder()

state_dict = torch.load(MODEL_PATH, map_location="cpu")

model.load_state_dict(state_dict)

model.eval()


Autoencoder(
  (encoder_conv): Sequential(
    (0): Conv2d(3, 4, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(4, 8, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=(2, 3), stride=(2, 3), padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(8, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
  )
  (flatten): Flatten(start_dim=1, end_dim=-1)
  (encoder_fc): Sequential(
    (0): Linear(in_features=7500, out_features=16, bias=True)
    (1): ReLU()
  )
  (decoder_fc): Sequential(
    (0): Linear(in_features=16, out_features=7500, bias=True)
    (1): ReLU()
  )
  (decoder_conv): Sequential(
    (0): Conv2d(12, 12, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): Upsample(scale_factor=(2.0, 3.0), mode='nearest')
    (3): Conv2d(12, 8, kernel_size=(3, 3), stride=(1, 1), pad

# Load images and labels from the manifest

Load validation and test images from manifest
Note:
Validation samples are physically stored in the test folder.
The manifest split column determines the logical partition.

In [5]:
def load_data_from_manifest(data_root: Path, manifest_csv: Path, split: str, target_size=(150, 100)):
    df = pd.read_csv(manifest_csv)
    df_split = df[df["split"] == split].reset_index(drop=True)

    images, labels, filenames = [], [], []

    for _, row in tqdm(df_split.iterrows(), total=len(df_split)):
        filename = str(row["filename"])
        label = int(row["label"])

        # Dateipfad: train/ bzw. test/ (validation liegt physisch in test/, aber Manifest sagt split=validation)
        folder = data_root / ("test" if split in ["test", "validation"] else "train")
        file_path = folder / filename

        if not file_path.exists():
            print(f"Missing: {file_path}")
            continue

        img = Image.open(file_path).convert("RGB").resize(target_size)
        img = np.array(img).astype("float32") / 255.0

        images.append(img)
        labels.append(label)
        filenames.append(filename)

    return np.array(images), np.array(labels), np.array(filenames)

In [6]:
# Load images

X_val, y_val, f_val = load_data_from_manifest(
    DATA_ROOT,
    MANIFEST_PATH,
    "validation",
    IMAGE_SIZE
)

X_test, y_test, f_test = load_data_from_manifest(
    DATA_ROOT,
    MANIFEST_PATH,
    "test",
    IMAGE_SIZE
)

print(X_val.shape)
print(X_test.shape)

  0%|          | 0/2500 [00:00<?, ?it/s]

100%|██████████| 2500/2500 [01:38<00:00, 25.32it/s] 


torch.Size([2500, 3, 100, 150])
torch.Size([2500, 3, 100, 150])


In [106]:
def select_best_f1_threshold(y_true, scores):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)

    # Edge case
    if len(thresholds) == 0:
        t = float(np.max(scores))
        return {
            "threshold": t,
            "validation_f1": 0.0,
            "validation_precision": 0.0,
            "validation_recall": 0.0
        }

    # F1 berechnen
    f1 = 2 * precision[:-1] * recall[:-1] / (precision[:-1] + recall[:-1] + 1e-12)

    best_idx = int(np.nanargmax(f1))

    return {
        "threshold": float(thresholds[best_idx]),
        "validation_f1": float(f1[best_idx]),
        "validation_precision": float(precision[best_idx]),
        "validation_recall": float(recall[best_idx]),
    }

In [107]:
def evaluate_at_threshold(y_true, scores, threshold):
    y_pred = (scores >= threshold).astype(int)

    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()

    return {
        "pr_auc": float(average_precision_score(y_true, scores)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "tn": int(tn),
        "fp": int(fp),
        "fn": int(fn),
        "tp": int(tp),
    }

# SVE - Segmented Vertical Error

In [108]:
def vertical_segment_scores(error_map: np.ndarray, n_segments: int, top_k: int):
    width = error_map.shape[2]

    boundaries = np.linspace(0, width, n_segments + 1, dtype=int)

    segment_scores = []
    for left, right in zip(boundaries[:-1], boundaries[1:]):
        if right <= left:
            continue

        segment_scores.append(
            error_map[:, :, left:right, :].mean(axis=(1, 2, 3))
        )

    segment_scores = np.stack(segment_scores, axis=1)

    sve_max = segment_scores.max(axis=1)

    top_k = min(top_k, segment_scores.shape[1])
    sve_topk = np.sort(segment_scores, axis=1)[:, -top_k:].mean(axis=1)

    return sve_max, sve_topk


def score_reconstructions(model, images, batch_size=32):
    # NHWC -> NCHW
    images_torch = (
        torch.from_numpy(images)
        .permute(0, 3, 1, 2)
        .float()
    )
    model.eval()
    recon_batches = []

    with torch.no_grad():
        for start in range(0, len(images_torch), batch_size):
            batch = images_torch[start:start + batch_size]
            recon = model(batch)
            recon_batches.append(
                recon.permute(0, 2, 3, 1).cpu().numpy()
            )

    recon = np.concatenate(recon_batches, axis=0)

    squared_error = (images - recon) ** 2
    absolute_error = np.abs(images - recon)

    score_data = {
        "global_mse": squared_error.mean(axis=(1, 2, 3)),
        "global_mae": absolute_error.mean(axis=(1, 2, 3)),
    }

    sve_7, _ = vertical_segment_scores(
        squared_error,
        n_segments=7,
        top_k=1
    )
    score_data["sve_7"] = sve_7

    sve_5, _ = vertical_segment_scores(
        squared_error,
        n_segments=5,
        top_k=1
    )   
    score_data["sve_5"] = sve_5
 
    return pd.DataFrame(score_data)

In [109]:
# ==========================================================
# Compute reconstruction-based anomaly scores
# ==========================================================
val_scores = score_reconstructions(model, X_val)
test_scores = score_reconstructions(model, X_test) 

## Robustness Analysis via Repeated Stratified 5-Fold Cross-Validation

Validation and test samples are merged into a common
evaluation pool. Thresholds are estimated on training folds
and evaluated on unseen folds using repeated stratified
cross-validation.

In [111]:
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

cv_results = []

val_df = val_scores.copy()
val_df["filename"] = f_val
val_df["label"] = y_val
val_df["split"] = "validation"

test_df = test_scores.copy()
test_df["filename"] = f_test
test_df["label"] = y_test
test_df["split"] = "test"

all_scores = pd.concat([val_df, test_df])
all_labels = all_scores["label"].values

for score_name in ["global_mse","global_mae","sve_5","sve_7"]:

    for fold, (train_idx, test_idx) in enumerate(
        cv.split(all_scores, all_labels),
        start=1
    ):

        train_fold = all_scores.iloc[train_idx]
        test_fold = all_scores.iloc[test_idx]

        train_labels = train_fold["label"].values
        test_labels = test_fold["label"].values

        selected = select_best_f1_threshold(
            train_labels,
            train_fold[score_name].values
        )

        threshold = selected["threshold"]

        metrics = evaluate_at_threshold(
            test_labels,
            test_fold[score_name].values,
            threshold
        )

        cv_results.append({
            "score": score_name,
            "cv_iteration": fold,
            "threshold": threshold,
            **metrics
        })

In [112]:
robustness_results_df = pd.DataFrame(cv_results)

robustness_summary_df = (
    robustness_results_df
    .groupby("score")
    .agg(
        pr_auc_mean=("pr_auc", "mean"),
        pr_auc_std=("pr_auc", "std"),
        f1_mean=("f1", "mean"),
        f1_std=("f1", "std"),
        precision_mean=("precision", "mean"),
        precision_std=("precision", "std"),
        recall_mean=("recall", "mean"),
        recall_std=("recall", "std"),
        threshold_mean=("threshold", "mean"),
        threshold_std=("threshold", "std"),
    )
    .round(3)
)
robustness_summary_df = robustness_summary_df.sort_values(
    "pr_auc_mean",
    ascending=False
).reset_index(drop=False)

#write results to table
ROBUSTNESS_RESULTS_PATH = (
    SVE_PATH /
    "ae_robustness_fold_results.csv"
)

robustness_results_df.to_csv(
    ROBUSTNESS_RESULTS_PATH,
    index=False
)

print(f"Wrote {ROBUSTNESS_RESULTS_PATH}")

robustness_summary_df

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\SVE\ae_robustness_fold_results.csv


,score,pr_auc_mean,pr_auc_std,f1_mean,f1_std,precision_mean,precision_std,recall_mean,recall_std,threshold_mean,threshold_std
0,sve_7,0.926,0.046,0.849,0.101,0.845,0.100,0.876,0.145,0.003,0.0
1,sve_5,0.907,0.072,0.861,0.071,0.845,0.100,0.889,0.101,0.002,0.0
2,global_mse,0.643,0.131,0.659,0.144,1.000,0.000,0.507,0.154,0.001,0.0
3,global_mae,0.539,0.140,0.589,0.161,0.880,0.146,0.462,0.166,0.014,0.0


## Save summary as LaTeX table 

In [113]:
score_labels = {
    "global_mae": "Global MAE",
    "global_mse": "Global MSE",
    "sve_5": "AE-SVE-5",
    "sve_7": "AE-SVE-7",
}

latex_lines = [
    r"\begin{tabular}{lrrrr}",
    r"\toprule",
    r"Method & Precision & Recall & F1-score & PR-AUC \\",
    r"\midrule",
]

for _, row in robustness_summary_df.iterrows():

    latex_lines.append(
        f"{score_labels[row['score']]} & "
        f"{row['precision_mean']:.3f} $\\pm$ {row['precision_std']:.3f} & "
        f"{row['recall_mean']:.3f} $\\pm$ {row['recall_std']:.3f} & "
        f"{row['f1_mean']:.3f} $\\pm$ {row['f1_std']:.3f} & "
        f"{row['pr_auc_mean']:.3f} $\\pm$ {row['pr_auc_std']:.3f} \\\\"
    )

latex_lines.extend([
    r"\bottomrule",
    r"\end{tabular}",
])

latex_table = "\n".join(latex_lines)

# Save
LATEX_PATH = SVE_PATH / "sve_nested_robustness_cv.tex"
LATEX_PATH.write_text(latex_table, encoding="utf-8")

print(f"Wrote {LATEX_PATH}")

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\SVE\sve_nested_robustness_cv.tex


# Save Metrics

In [115]:
SCORES_PATH = SCORE_DIR / "ae_scores_broach_dataset_robustness_cv.csv"
THRESHOLD_PATH = THRESHOLD_DIR / "ae_thresholds_broach_dataset_robustness_cv.json"
METRICS_PATH = TABLE_DIR / "metrics_broach_dataset_ae_robustness_cv.csv"


# --- Speichere val_scores + test_scores als CSV ---
all_scores = pd.concat([val_df, test_df], ignore_index=True)
all_scores.to_csv(SCORES_PATH, index=False)
print(f'Wrote {SCORES_PATH}')

score_columns = [
    "global_mse",
    "global_mae",
    "sve_5",
    "sve_7",
]

results = []
thresholds = {}

for score_name in score_columns:
    # --- Threshold (Validation!)
    selected = select_best_f1_threshold(
        y_val,
        val_scores[score_name].values
    )
    threshold = selected["threshold"]
    thresholds[score_name] = selected  

    # --- Test Evaluation ---
    test_metrics = evaluate_at_threshold(
        y_test,
        test_scores[score_name].values,
        threshold
    )
    results.append({
        "dataset": 'broach_dataset',  
        "method": 'cnn_ae',      
        "score": score_name,
        "threshold": threshold,
        **test_metrics
    })

with THRESHOLD_PATH.open('w') as f:
    json.dump(thresholds, f, indent=2)
print(f'Wrote {THRESHOLD_PATH}')

# --- Speichere Metriken als CSV ---
pd.DataFrame(results).to_csv(METRICS_PATH, index=False)
print(f'Wrote {METRICS_PATH}')

# Ausgabe für die Notebook-Zelle
final_results_df = pd.DataFrame(results).sort_values("pr_auc", ascending=False).reset_index(drop=True)
final_results_df

Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\scores\ae_scores_broach_dataset_robustness_cv.csv
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\thresholds\ae_thresholds_broach_dataset_robustness_cv.json
Wrote C:\Users\Zeleny\Documents\01_VSCode\spectrogram-anomaly-ae\reports\tables\metrics_broach_dataset_ae_robustness_cv.csv


,dataset,method,score,threshold,pr_auc,f1,precision,recall,tn,fp,fn,tp
0,broach_dataset,cnn_ae,sve_7,0.003109,0.979270,0.850000,0.944444,0.772727,2477,1,5,17
1,broach_dataset,cnn_ae,sve_5,0.001871,0.945071,0.909091,0.909091,0.909091,2476,2,2,20
2,broach_dataset,cnn_ae,global_mse,0.001131,0.592931,0.666667,1.000000,0.500000,2478,0,11,11
3,broach_dataset,cnn_ae,global_mae,0.013949,0.465159,0.516129,0.888889,0.363636,2477,1,14,8
